# Exercise 1 — The simplest simulation with `o2-sim`

`o2-sim` creates the ALICE geometry, generates primary particles, transports them through the detector with Geant4 (or Geant3/FLUKA), and writes hits and the MC kinematics.

This exercise covers:
- **1a** — Pythia8 pp events through the full detector
- **1b** — reading the kinematics output
- **1c** — making it faster (generator only, fewer modules)
- **1d** — a box generator shooting Λc⁺

In [ ]:
from swan_env import *

In [ ]:
ex1 = workdir("ex1")
os.environ["EX1"] = str(ex1)

## 1a. Pythia8 pp events, full detector

Generate 5 minimum-bias pp events (`pythia8pp`, √s = 14 TeV) and transport them with Geant4 using 4 workers.
This takes a few minutes, mostly for initialisation.

> ⚠️ This needs the AliEn token from `00_setup.ipynb`, because the detector alignment is read from CCDB.
> Without a token, the worker stops with `TGrid::Connect returned nullptr`. As a workaround you can add
> `--configKeyValues "align-geom.mDetectors=none"` to simulate the ideal, non-aligned geometry.

In [ ]:
%%bash
cd $EX1
time o2exec o2-sim -n 5 -g pythia8pp -j 4 --seed 42 > sim_pythia.log 2>&1
tail -5 sim_pythia.log

In [ ]:
%%bash
cd $EX1
ls -lh

Output files:

| File | Content |
|---|---|
| `o2sim_Kine.root` | MC kinematics: primary and secondary particles (`o2::MCTrack`) |
| `o2sim_MCHeader.root` | event-level information (impact parameter, generator info, …) |
| `o2sim_geometry.root`, `o2sim_grp.root` | geometry and run parameters used |
| `o2sim_<DET>.root` | hits of each detector |
| `o2sim_serverlog`, `o2sim_workerlog0`, `o2sim_mergerlog` | logs of the event server, transport workers and hit merger |

**❓ Questions**
1. How many hit files were produced? Which detectors are simulated by default?
2. Find the Pythia8 initialisation in `o2sim_serverlog`. Which beams and energy were used?

In [ ]:
%%bash
cd $EX1
grep -m5 -E 'Beams:(idA|idB|eCM)' o2sim_serverlog

## 1b. A look at the kinematics

The kinematics tree `o2sim` has one entry per event. Each entry holds a `std::vector<o2::MCTrack>`.
The ROOT macros run inside the O2 environment (`o2exec root ...`), so the O2 classes are known.

In [ ]:
%%bash
cd $EX1
o2exec root -l -b -q "$SIMTUT_PROJECT/macros/kineOverview.C(\"o2sim_Kine.root\",\"kine_pythia.png\")"

In [ ]:
show(ex1 / "kine_pythia.png")

**❓** The production vertex y distribution has peaks. Which detector structures do they match?

## 1c. Make it faster

- `--noGeant`: generator output only, no transport. This takes seconds.
- `-m PIPE ITS TPC`: only some modules in the geometry.

In [ ]:
%%bash
cd $EX1
echo "### generator only";  time o2exec o2-sim -n 100 -g pythia8pp --noGeant -o gen_only > gen_only.log 2>&1
echo "### PIPE+ITS+TPC";    time o2exec o2-sim -n 5 -g pythia8pp -j 4 -m PIPE ITS TPC -o itstpc > itstpc.log 2>&1
ls -lh *_Kine.root

**❓** Compare the time per event and the size of the kinematics files with 1a. Why is `gen_only_Kine.root` so small even with 100 events?

## 1d. A box generator: Λc⁺ gun

`boxgen` shoots `BoxGun.number` particles of type `BoxGun.pdg`, uniform in η, p and φ.
Here: 5 Λc⁺ (PDG 4122) per event, |η| < 0.8, 1 < p < 10 GeV/c.

In [ ]:
%%bash
cd $EX1
time o2exec o2-sim -n 10 -g boxgen -j 4 -m PIPE ITS TPC -o lambdac \
  --configKeyValues "BoxGun.pdg=4122;BoxGun.number=5;BoxGun.eta[0]=-0.8;BoxGun.eta[1]=0.8;BoxGun.prange[0]=1.;BoxGun.prange[1]=10.;BoxGun.phirange[0]=0.;BoxGun.phirange[1]=360." \
  > lambdac.log 2>&1
tail -3 lambdac.log

In [ ]:
%%bash
cd $EX1
o2exec root -l -b -q "$SIMTUT_PROJECT/macros/kineDaughters.C(\"lambdac_Kine.root\",4122)"

**❓ Questions**
1. What are the most frequent direct decay products of the Λc⁺? Compare with the PDG branching ratios.
2. Change the gun to π⁺ (`BoxGun.pdg=211`) and compare the number of tracks per event, i.e. the secondaries produced in the material.

> 💡 Recent O2 versions can also sample uniformly in **pT and rapidity**: add `BoxGun.sampleYAndPt=true`, and `eta`/`prange` are then read as y and pT ranges.

In [ ]:
%%bash
# your turn: pi+ gun
cd $EX1

## Save your results to CERNBox

In [ ]:
%%bash
mkdir -p ~/simtutorial_results/ex1
cp $EX1/*.png $EX1/*_Kine.root ~/simtutorial_results/ex1/ && ls ~/simtutorial_results/ex1

Next: **`02_hybrid_workflow.ipynb`**